# exp-02 — does splitting a check skill into a DAG degrade it?

Reads the runs committed under `experiments/runs/exp-02-delegation-depth/`,
scores every leaf that is not scored yet, and compares each split arrangement
against the monolith.

**This notebook spends nothing.** The sessions are produced by

```bash
python experiments/exp-02-delegation-depth/run.py
```

which is 2,280 sessions and most of a day. Keeping the run out of here is what
makes the analysis re-executable: anyone can rerun this against the committed
predictions without paying for them again.

Scoring *is* done here, because the scored `analysis.json` is derived from the
manifest as it stands now, and a rollup cached at run time would be one that
nothing invalidates when a threshold moves.

The claims this is evidence for live in
[`thinking/experiments/exp-02-delegation-depth.md`](../../thinking/experiments/exp-02-delegation-depth.md).
Nothing here decides whether the experiment succeeded.

## The four arrangements

Every check is built from three blocks — **A** panel identification, **B**
panel classification, **C** the check itself — and the arrangements differ only
in how many skills those blocks are spread across.

| arrangement | arm directory | skills |
|---|---|---|
| `A\|B\|C` | `pinned` | 1 — the monolith, and the baseline |
| `A <- B\|C` | `<check>@v2` | 2 |
| `A\|B <- C` | `<check>@v3` | 2 |
| `A <- B <- C` | `classify-panels@v2-<check>@v3` | 3 |

The prose is word-identical across all four; what differs is where it lives.
The evaluation contracts sit above the version directories, so every
arrangement of one check is scored by the same schema and the same gold.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from soda_mmqc.core.run_layout import BASELINE_ARM, iter_leaves
from soda_mmqc.core.scoring import ANALYSIS_FILENAME, score_check
from soda_mmqc.reporting import (
    arm_contrast,
    layer1_counts_by_example,
    layer_s_counts_by_example,
    load_run_root,
    non_response_counts,
    rate_contrast,
    scores_frame,
)

CHECKLIST = "fig-checklist-exp02"
RUNS = Path("../../experiments/runs/exp-02-delegation-depth")
CHECKS = ("micrograph-scale-bar", "individual-data-points", "error-bars-defined")

#: Margins, fixed in the note before the run. One per layer, because the
#: layers sit on different scales and are measured to different precision.
DELTA = {"layer S": 0.0125, "layer 1": 0.02, "layer 2": 0.02}

#: Seeded so the intervals are reproducible; an interval a reader cannot
#: recompute is not one an experiment can preregister.
SEED = 20260926


def arrangement(arm: str, check: str) -> str:
    """Name an arm by its topology rather than by what moved off the pin."""
    if arm == BASELINE_ARM:
        return "A|B|C"
    if arm == f"{check}@v2":
        return "A <- B|C"
    if arm == f"{check}@v3":
        return "A|B <- C"
    if arm == f"classify-panels@v2-{check}@v3":
        return "A <- B <- C"
    return arm


SPLIT = ("A <- B|C", "A|B <- C", "A <- B <- C")
print(RUNS.resolve())

## Score every leaf that is not scored yet

A leaf is one `<check>/<arm>/rep-NN/` directory, and `score_check` takes
exactly one. Aggregating across arms and replicates is this notebook's job,
not the harness's — which is what keeps the evaluator general.

This is idempotent: a leaf that already has an `analysis.json` is skipped. That
is a convenience and a trap — if the predictions under a leaf are replaced, its
stale analysis will be kept and silently scored as though nothing changed. The
cell below checks for that rather than trusting it.

In [ ]:
stale = []
for check in CHECKS:
    for arm, replicate, leaf in iter_leaves(RUNS / check):
        analysis = leaf / ANALYSIS_FILENAME
        if not analysis.is_file():
            score_check(CHECKLIST, check, leaf, save=True)
            print(f"scored {check} / {arm} / rep-{replicate:02d}")
            continue
        newest = max(
            (p.stat().st_mtime for p in leaf.rglob("prediction.json")),
            default=0.0,
        )
        if newest > analysis.stat().st_mtime:
            stale.append(leaf)

if stale:
    raise RuntimeError(
        "predictions are newer than their analysis in "
        f"{len(stale)} leaf/leaves; delete those analysis.json and re-run "
        f"this cell:\n  " + "\n  ".join(str(p) for p in stale)
    )
print("all leaves scored, none stale")

## Endpoint 0 — did delegation actually happen?

Read this before any score. A prediction that looks right says nothing about
how it was reached, and an arm whose shared skill never fired is not the
arrangement it is labelled as.

`skill_trace.json` records every skill the session invoked. The pilot found
delegation to be partial and check-dependent — a shared skill gets invoked when
the caller cannot proceed without it, and skipped when the caller's own prose
already answers the question — so this is **reported, not a gate**. The note
fixes how to read it: a non-inferior arm with partial invocation means
decomposition is *safe but optional*, and a degraded arm is read together with
its rate, since an arm that mostly did not delegate is weak evidence that
delegating is what hurt.

In [ ]:
import json

EXPECTED = {
    "A|B|C": set(),
    "A <- B|C": {"identify-panels"},
    "A|B <- C": {"classify-panels"},
    "A <- B <- C": {"classify-panels", "identify-panels"},
}

rows = []
for check in CHECKS:
    for arm, replicate, leaf in iter_leaves(RUNS / check):
        want = EXPECTED[arrangement(arm, check)]
        for trace in leaf.rglob("intermediates/skill_trace.json"):
            try:
                entries = json.loads(trace.read_text(encoding="utf-8"))
            except (OSError, json.JSONDecodeError):
                continue
            seen = {e.get("skill") for e in entries if isinstance(e, dict)}
            rows.append({
                "check": check,
                "arrangement": arrangement(arm, check),
                "replicate": replicate,
                "full_chain": want.issubset(seen),
                "n_skills": len(seen),
            })

traces = pd.DataFrame(rows)
invocation = (
    traces.groupby(["arrangement", "check"])["full_chain"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "rate", "sum": "chained", "count": "sessions"})
)
invocation

In [ ]:
# The headline rate per arrangement, which is what endpoint 0 reports.
by_arrangement = (
    traces.groupby("arrangement")["full_chain"]
    .agg(["mean", "sum", "count"])
    .rename(columns={"mean": "rate", "sum": "chained", "count": "sessions"})
    .reindex(["A|B|C", *SPLIT])
)
by_arrangement

## One row per property, per replicate, per example

`scores_frame` keeps both axes of variation — which example, which replicate —
and never collapses `property` into a row. Every layer-2 statistic below groups
this frame, and none of them group across `property`: `panel_label` and
`micrograph` measure different things, and a number mixing them hides an
arrangement that improves one while degrading the other.

In [ ]:
frames, layer_s, layer_1 = [], [], []
for check in CHECKS:
    runs = load_run_root(RUNS / check, checklist=CHECKLIST, check=check)
    frames.append(scores_frame(runs))
    layer_s.append(layer_s_counts_by_example(runs))
    layer_1.append(layer1_counts_by_example(runs))

scores = pd.concat(frames, ignore_index=True)
layer_s = pd.concat(layer_s, ignore_index=True)
layer_1 = pd.concat(layer_1, ignore_index=True)

for frame in (scores, layer_s, layer_1):
    frame["arrangement"] = [
        arrangement(a, c) for a, c in zip(frame["arm"], frame["check"])
    ]

print(f"{len(scores)} score rows, {scores['check'].nunique()} checks, "
      f"{scores['replicate'].nunique()} replicates, "
      f"{scores['arrangement'].nunique()} arrangements")
scores.head()

## Non-response, before any mean

A session that answers `outputs: []` scores as a fully missing row set rather
than being excluded — an arrangement so lossy the model returns nothing is a
worse result, not an absent one.

It also interacts with layer 2 in the direction that misleads: an arm that
answers nothing has no applicable instances either, so it contributes nothing
to the layer-2 mean and can appear to score **better** by saying less.

In [ ]:
non_response = pd.concat(
    [
        non_response_counts(
            load_run_root(RUNS / check, checklist=CHECKLIST, check=check)
        )
        for check in CHECKS
    ],
    ignore_index=True,
)
non_response["arrangement"] = [
    arrangement(a, c) for a, c in zip(non_response["arm"], non_response["check"])
]
(
    non_response.groupby(["check", "arrangement"])["empty"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=["A|B|C", *SPLIT])
)

## The verdict rule

Each split arrangement is compared against the monolith, paired by example and
bootstrapped over the 38 examples — the clustering unit, because rows inside
one figure share a figure, a caption and a curator.

- **Non-inferior** when the lower bound of the 95% CI lies above **−δ**
- **Degraded** when the interval lies entirely below −δ
- **Inconclusive** when it straddles −δ, including when it is simply wider
  than the margin

Three comparisons, each running the gate sequence independently. They are three
separate claims about three ways of splitting one skill, not a family from which
the best is chosen, so nothing is corrected across them — and all three are
reported whatever they say.

In [ ]:
def verdict(low, high, delta):
    """Read one interval against one margin."""
    if pd.isna(low) or pd.isna(high):
        return "no data"
    if low > -delta:
        return "non-inferior"
    if high < -delta:
        return "degraded"
    return "inconclusive"


def rate_gate(frame, *, category, numerator, denominator, group, delta, label):
    """One rate endpoint, for every split arrangement against the monolith."""
    out = []
    for check in CHECKS:
        sub = frame[frame["check"] == check]
        base = sub[sub["arrangement"] == "A|B|C"]
        for split in SPLIT:
            pair = pd.concat([base, sub[sub["arrangement"] == split]])
            if pair["arrangement"].nunique() < 2:
                continue
            contrast = rate_contrast(
                pair.drop(columns=["arm"]).rename(columns={"arrangement": "arm"}),
                baseline="A|B|C",
                variant=split,
                category=category,
                numerator=numerator,
                denominator=denominator,
                group=group,
                seed=SEED,
            )
            contrast.insert(0, "arrangement", split)
            contrast.insert(0, "endpoint", label)
            out.append(contrast)
    result = pd.concat(out, ignore_index=True)
    result["half_width"] = (result["ci_high"] - result["ci_low"]) / 2
    result["verdict"] = [
        verdict(lo, hi, delta)
        for lo, hi in zip(result["ci_low"], result["ci_high"])
    ]
    return result

## Gate 1 — layer S: did the arrangement return the right row sets?

`correct_row` as a fraction of gold rows. This is where delegating panel
enumeration into `identify-panels` would show: a panel dropped at the boundary
is a row that never reaches layer 1 or layer 2.

δ = 0.0125, tighter than the other gates because layer S sits near its ceiling
and the largest movement check prose has produced there is smaller.

In [ ]:
gate1 = rate_gate(
    layer_s,
    category="outcome",
    numerator=("correct_row",),
    denominator=("correct_row", "missing_row"),
    group=("check", "list_key"),
    delta=DELTA["layer S"],
    label="layer S",
)
gate1[[
    "arrangement", "check", "list_key", "difference", "ci_low", "ci_high",
    "half_width", "n_examples", "paired_fraction", "verdict",
]]

## Gate 2 — layer 1: did it judge applicability the same way?

`correct_applicable` + `correct_NA` as a fraction of all profiled instances.
This is where delegating classification into `classify-panels` would show —
whether a panel is a micrograph, or a plot, is what decides applicability.

Read it beside the non-response counts above: an arm that withholds more has
fewer instances left for layer 2, which is the confound this experiment is most
exposed to.

In [ ]:
gate2 = rate_gate(
    layer_1,
    category="layer1",
    numerator=("correct_applicable", "correct_NA"),
    denominator=None,
    group=("check",),
    delta=DELTA["layer 1"],
    label="layer 1",
)
gate2[[
    "arrangement", "check", "difference", "ci_low", "ci_high",
    "half_width", "n_examples", "paired_fraction", "verdict",
]]

## Gate 3 — layer 2: the prose that stayed in the leaf

`mean_score` per property, on applicable instances, paired by example with
replicates averaged per `(arm, example)` first — they are resamples of one
measurement, so they reduce its noise rather than adding rows.

Per property, never pooled: pooling mixes `panel_label` with `micrograph`, so
an arrangement that improves one and degrades the other reads as nothing
happening.

`arm_contrast` reports an analytic `se` rather than a bootstrap interval, so
the interval here is `difference ± 1.96·se` — the same form the margins were
sized against in the note.

In [ ]:
rows = []
for check in CHECKS:
    sub = scores[scores["check"] == check]
    base = sub[sub["arrangement"] == "A|B|C"]
    for split in SPLIT:
        pair = pd.concat([base, sub[sub["arrangement"] == split]])
        if pair["arrangement"].nunique() < 2:
            continue
        contrast = arm_contrast(
            pair.drop(columns=["arm"]).rename(columns={"arrangement": "arm"}),
            baseline="A|B|C",
            variant=split,
        )
        contrast.insert(0, "arrangement", split)
        rows.append(contrast)

gate3 = pd.concat(rows, ignore_index=True)
gate3["half_width"] = 1.96 * gate3["se"].astype(float)
gate3["ci_low"] = gate3["difference"].astype(float) - gate3["half_width"]
gate3["ci_high"] = gate3["difference"].astype(float) + gate3["half_width"]
gate3["resolves_delta"] = gate3["half_width"] < DELTA["layer 2"]
gate3["verdict"] = [
    verdict(lo, hi, DELTA["layer 2"])
    for lo, hi in zip(gate3["ci_low"], gate3["ci_high"])
]
gate3.sort_values("difference")[[
    "arrangement", "check", "property", "difference", "ci_low", "ci_high",
    "half_width", "resolves_delta", "n_examples", "paired_fraction", "verdict",
]]

### Which properties could resolve the margin

A property can only support a 0.02 claim if its half-width is smaller than
0.02; otherwise even an observed difference of exactly zero leaves an interval
whose lower end falls below −0.02, whatever the data say.

Sized from exp-01 at five replicates, all 17 were expected to clear it. This
reports what actually happened, so a reader can see where that prediction held.
A property predicted to clear the floor that turns out wider is reported as
uninformative, not quietly counted.

In [ ]:
(
    gate3.groupby(["arrangement", "resolves_delta"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={True: "resolves 0.02", False: "below the floor"})
)

## The three claims, side by side

One row per comparison per gate. The gates are conditional — layer 2 is scored
only on instances layer 1 judged applicable, and layer 1 only where layer S
found a row — so an arrangement that fails an early gate makes the later ones
comparisons between different populations rather than between arrangements.

Endpoint 0 sits beside them rather than in front: it qualifies what a verdict
means, it does not decide whether the verdict counts.

In [ ]:
def summarise(frame, endpoint, delta):
    g = frame.groupby("arrangement")["verdict"].value_counts().unstack(fill_value=0)
    g.insert(0, "delta", delta)
    g.insert(0, "endpoint", endpoint)
    return g

summary = pd.concat([
    summarise(gate1, "layer S", DELTA["layer S"]),
    summarise(gate2, "layer 1", DELTA["layer 1"]),
    summarise(gate3[gate3["resolves_delta"]], "layer 2", DELTA["layer 2"]),
])
summary = summary.join(by_arrangement["rate"].rename("invocation"))
summary

## What this does not do

It does not decide whether the experiment succeeded. There is no overall score
and no significance test beyond the preregistered intervals, because what
counts as a real difference is the experiment's claim and belongs in
[`thinking/experiments/exp-02-delegation-depth.md`](../../thinking/experiments/exp-02-delegation-depth.md),
where the reasoning can be read and disagreed with.

In particular, the four-cell reading of endpoint 0 against the score endpoints —
whether partial invocation with intact scores means *decomposition is safe but
optional* — is an interpretation fixed in the note before the run, not a
conclusion this notebook computes.